# E2 — Esercitazione: descrivere e unire (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/E2_descrittive_unione/E2_soluzioni.ipynb)

## Come funziona un'esercitazione

Rispetto alle lezioni, qui hai **meno indicazioni**: per ogni passo trovi l'obiettivo e un **controllo** per verificare di esserci arrivato. Cosa cercare, sta a te (Google, Gemini, le dispense, i notebook delle lezioni).

- 👥 Si lavora in **gruppi da 3**, con tre ruoli che **ruotano a ogni passo**: chi **scrive** (condivide lo schermo e scrive il codice), chi **cerca** (Google, Gemini, dispense), chi **controlla** (il risultato ha senso? il numero torna?).
- Se il controllo non torna, **non andare avanti**: un errore all'inizio si porta dietro tutto il resto.
- **File → Salva una copia in Drive** per non perdere il lavoro.

## Contesto

Una fondazione ti chiede una breve nota sulle imprese del campione: come si distribuiscono tra le macroaree, come è andato il fatturato negli anni, chi investe di più in ricerca e sviluppo. Hai gli strumenti della L4 (raggruppare, disegnare) e della L5 (unire).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/imprese_pulito.xlsx")
regioni = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/regioni.xlsx")

### 1. Unisci

Aggiungi a ogni riga di `df` la macroarea della sua regione.

✅ *Controllo:* 2.500 righe e nessuna macroarea mancante.

In [ ]:
regioni["regione"] = regioni["regione"].replace({
    "Emilia Romagna": "Emilia-Romagna",
    "Trentino-Alto Adige/Suedtirol": "Trentino-Alto Adige",
})
df = df.merge(regioni, on="regione", how="left", validate="many_to_one")
print(df.shape, df["macroarea"].isna().sum())

### 2. Il quadro per macroarea

Costruisci una tabella con una riga per macroarea e queste colonne: numero di imprese diverse, fatturato mediano, produttività mediana (fatturato per addetto).

✅ *Controllo:* 5 righe; le imprese sommano a 500.

In [ ]:
df["produttivita"] = df["fatturato"] / df["addetti"]
tab = df.groupby("macroarea").agg(
    imprese=("id_impresa", "nunique"),
    fatturato_mediano=("fatturato", "median"),
    produttivita_mediana=("produttivita", "median"),
)
print(tab["imprese"].sum())
tab.round(1)

### 3. Nel tempo, per macroarea

Disegna un grafico con **una linea per macroarea**: fatturato mediano per anno. Titolo e unità di misura.

✅ *Controllo:* 5 linee, e il calo del 2020 si vede in tutte.

In [ ]:
df.pivot_table(index="anno", columns="macroarea", values="fatturato", aggfunc="median").plot(marker="o")
plt.title("Fatturato mediano per macroarea")
plt.ylabel("Migliaia di euro")
plt.show()

### 4. Ricerca e sviluppo

Calcola l'**intensità di R&S** (spesa in R&S divisa per fatturato) di ogni riga, poi la mediana per settore. Disegnala con un grafico a barre orizzontali ordinato.

In [ ]:
df["intensita_rs"] = df["spesa_rs"] / df["fatturato"]
df.groupby("settore")["intensita_rs"].median().sort_values().plot(kind="barh")
plt.title("Intensità di R&S mediana per settore")
plt.xlabel("Spesa in R&S / fatturato")
plt.show()

*Risposta:* Le differenze tra settori sono piccole: nel campione l'intensità mediana è tra il 2 e il 3,5% ovunque. Anche "non c'è una differenza" è un risultato, purché sia detto chiaramente.

### 5. La tua domanda

Fai **una domanda tua** a questi dati (per esempio: le imprese più vecchie sono più grandi? quale regione ha più imprese della manifattura?). Rispondi con una tabella o un grafico, e due frasi.

In [ ]:
df["eta"] = 2023 - df["data_costituzione"].dt.year
df["classe_eta"] = pd.cut(df["eta"], bins=[0, 10, 25, 60], labels=["<10 anni", "10-25 anni", ">25 anni"])
df[df["anno"] == 2023].groupby("classe_eta", observed=True)["addetti"].median()

*Risposta:* Esempio: tra le imprese del campione l'età non sembra legata alla dimensione: gli addetti mediani sono simili nelle tre classi di età. (I dati sono sintetici: nella realtà la relazione esiste.)

## La nota per la fondazione

Scrivi **cinque righe** per la fondazione con quello che hai trovato. Per ogni numero che citi, chiediti: su quante imprese si basa?

*Risposta:* Il campione conta 500 imprese, distribuite in tutte le macroaree. Il fatturato mediano è simile tra macroaree (tra circa 1.300 e 1.500 migliaia di euro). In tutte, il 2020 segna un calo di circa un quarto, recuperato nel 2021. L'intensità di R&S mediana è tra il 2 e il 3,5% in tutti i settori. Attenzione: alcune regioni contano pochissime imprese, quindi i confronti tra singole regioni sono fragili.